## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28*28, 100]), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([100]), dtype=tf.float32)
        self.W2 = tf.Variable(tf.random.normal([100, 10]), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28*28])  # 将输入数据展平
        h1 = tf.matmul(x, self.W1) + self.b1  # 第一个全连接层
        h1_relu = tf.nn.relu(h1)  # ReLU 激活函数
        logits = tf.matmul(h1_relu, self.W2) + self.b2  # 第二个全连接层
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 93.153915 ; accuracy 0.08431666
epoch 1 : loss 88.098145 ; accuracy 0.0854
epoch 2 : loss 84.031395 ; accuracy 0.08595
epoch 3 : loss 80.60637 ; accuracy 0.08681667
epoch 4 : loss 77.63542 ; accuracy 0.08766667
epoch 5 : loss 74.99823 ; accuracy 0.08873333
epoch 6 : loss 72.626564 ; accuracy 0.090116665
epoch 7 : loss 70.4848 ; accuracy 0.091583334
epoch 8 : loss 68.54636 ; accuracy 0.093783334
epoch 9 : loss 66.78918 ; accuracy 0.09601667
epoch 10 : loss 65.18909 ; accuracy 0.09816667
epoch 11 : loss 63.722874 ; accuracy 0.100483336
epoch 12 : loss 62.373043 ; accuracy 0.10255
epoch 13 : loss 61.12214 ; accuracy 0.10476667
epoch 14 : loss 59.955173 ; accuracy 0.107366666
epoch 15 : loss 58.86054 ; accuracy 0.11001667
epoch 16 : loss 57.82535 ; accuracy 0.112616666
epoch 17 : loss 56.838886 ; accuracy 0.1153
epoch 18 : loss 55.894566 ; accuracy 0.11793333
epoch 19 : loss 54.986233 ; accuracy 0.120516665
epoch 20 : loss 54.1081 ; accuracy 0.12321667
epoch 21 : loss 53.256